In [4]:
import pandas as pd
import music21 as m21
from fractions import Fraction
from harmonic_inference.data.data_types import ChordType, PitchType, KeyMode
from harmonic_inference.utils.harmonic_utils import get_scale_degree_from_interval, get_pitch_from_string
import harmonic_inference.utils.harmonic_constants as hc
from pathlib import Path
import os

In [29]:
# dicts

chord_types = {
"dominant-seventh": ChordType.MAJ_MIN7,
"major": ChordType.MAJOR,
"major-seventh": ChordType.MAJ_MAJ7,
"diminished": ChordType.DIMINISHED,
"augmented": ChordType.AUGMENTED,
"diminished-seventh": ChordType.DIM7,
"minor": ChordType.MINOR,
"minor-seventh": ChordType.MIN_MIN7,
"half-diminished-seventh": ChordType.HALF_DIM7,
"major-sixth": ChordType.MAJOR, # not defined in ChordType class
"minor-sixth": ChordType.MINOR, # Not defined in ChordType class
}


CHORD_TYPES_TO_STRING_READABLE = {
    ChordType.MAJ_MIN7: "D7",
    ChordType.MAJOR: "M",
    ChordType.MAJ_MAJ7: "M7",
    ChordType.DIMINISHED: "d",
    ChordType.AUGMENTED: "a",
    ChordType.DIM7: "d7",
    ChordType.MINOR: "m",
    ChordType.MIN_MIN7: "m7",
    ChordType.HALF_DIM7: "h7",
}

In [41]:
def process_score(score):

    key_changes = [] 
    
    for k in score.flat.getElementsByClass(m21.key.Key):  # actual key (major / minor)

        key_changes.append((k.offset, k))  
    if not key_changes:
        for ks in score.flat.getElementsByClass(m21.key.KeySignature): # count accidentals 

            key_changes.append((ks.offset, ks.asKey()))  # score.analyze("key")??



    key_changes.sort(key=lambda x: x[0])  

    # return the actual music21 Key object, not just the string
    def get_key_obj_at_offset(offset):  
        for ks_offset, k_obj in reversed(key_changes):  
            if offset >= ks_offset:  
                return k_obj
        return m21.key.Key('C')  # C as default Key object

    # format key str
    def format_key_string(k_obj):
        tonic_name = k_obj.tonic.name
        mode = k_obj.mode
        key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
        return key_str.replace('-', 'b').replace('#', '+')  

    # get offset length in score (in quarter note length) 

    piece_len = score.flat.highestOffset


    rows = []

    for harmony in score.flat.getElementsByClass("Harmony"):

        on = Fraction(harmony.offset)

        current_key_obj = get_key_obj_at_offset(on)
        key_str = format_key_string(current_key_obj)
        key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

        if hasattr(harmony, "root") and harmony.root() is not None:
            chord_root_string = harmony.root().name.replace("-", "b")
        else:
            chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

        chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC) # not replacing + or - with # and b will make the code crash
        key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
        degree = get_scale_degree_from_interval(
            chord_root_int - key_tonic_int,
            key_mode,
            PitchType.TPC,
        )

        harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
        chord_type_enum = chord_types.get(harmony_type_str, None)

        if chord_type_enum is not None:
            chord_type = CHORD_TYPES_TO_STRING_READABLE[chord_type_enum]

        else:
            chord_type = "unknown"
        # chord_type = chord_types.get(harmony_type_str, "unknown")

        # chord_type = str(chord_type) # make it a string


        inv = 0

        rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there

    # Create DataFrame WITHOUT column names

    df = pd.DataFrame(rows) # columns = ["on", "off", "key", "degree", "type", "inv"]

    # Shift the 'on' time of the next chord to become the "off" time of the current chord
    df[1] = df[0].shift(-1)

    # Handle the last chord's "off" time using its original duration since shift leaves None

    if not df.empty: # ensures the code only runs if score contains chords

        df.iloc[-1, 1] = Fraction(piece_len) # last_harmony.duration.quarterLength

    return df




In [42]:
process_score(m21.converter.parse("choro_corpus/with_chords/score_12248-Gloria-Bonfiglio_de_Oliveira.xml"))

,0,1,2,3,4,5
0,0,3,G,VI,m,0
1,3,6,G,III,D7,0
2,6,9,G,V,m,0
3,9,12,G,VI,D7,0
4,12,15,G,II,m,0
...,...,...,...,...,...,...
90,273,276,C,IV,M,0
91,276,279,C,I,M,0
92,279,282,C,V,D7,0
93,282,285,C,I,M,0


In [ ]:
# process entire directory

input_dir = Path("choro_corpus/with_chords")
output_dir = Path("csv_from_xml")

output_dir.mkdir(exist_ok=True)

for file in input_dir.glob("*.xml"):

    try:
        score = m21.converter.parse(file)

        df = process_score(score)

        output_file = output_dir / f"{file.stem}.csv"

        df.to_csv(
            output_file,
            index=False,
            header=False
        )
    except Exception as e: 
        print(f"Failed on {file.name}: {e}")

Failed on score_2802-Belezas_do_Recife-Misael_Domingues.xml: 'N.C.'


In [ ]:

key_changes = [] 
   
for k in score.flat.getElementsByClass(m21.key.Key):  # actual key (major / minor)

    key_changes.append((k.offset, k))  
if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature): # count accidentals 

        key_changes.append((ks.offset, ks.asKey()))  # score.analyze("key")??



key_changes.sort(key=lambda x: x[0])  

# return the actual music21 Key object, not just the string
def get_key_obj_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            return k_obj
    return m21.key.Key('C')  # C as default Key object

# format key str
def format_key_string(k_obj):
    tonic_name = k_obj.tonic.name
    mode = k_obj.mode
    key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
    return key_str.replace('-', 'b').replace('#', '+')  

# get offset length in score (in quarter note length) 

piece_len = score.flat.highestOffset


rows = []

for harmony in score.flat.getElementsByClass("Harmony"):

    on = Fraction(harmony.offset)

    current_key_obj = get_key_obj_at_offset(on)
    key_str = format_key_string(current_key_obj)
    key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

    if hasattr(harmony, "root") and harmony.root() is not None:
        chord_root_string = harmony.root().name.replace("-", "b")
    else:
        chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

    chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC) # not replacing + or - with # and b will make the code crash
    key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
    degree = get_scale_degree_from_interval(
        chord_root_int - key_tonic_int,
        key_mode,
        PitchType.TPC,
    )

    harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
    chord_type = chord_types.get(harmony_type_str, "unknown")

    chord_type = str(chord_type) # make it a string


    inv = 0

    rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there

# Create DataFrame WITHOUT column names

df = pd.DataFrame(rows) # columns = ["on", "off", "key", "degree", "type", "inv"]

# Shift the 'on' time of the next chord to become the "off" time of the current chord
df[1] = df[0].shift(-1)

# Handle the last chord's "off" time using its original duration since shift leaves None

if not df.empty: # ensures the code only runs if score contains chords
    last_harmony = score.flat.getElementsByClass("Harmony")[-1]

    df.iloc[-1, 1] = Fraction(piece_len) # last_harmony.duration.quarterLength

df




,0,1,2,3,4,5
0,0,3,G,VI,ChordType.MINOR,0
1,3,6,G,III,ChordType.MAJ_MIN7,0
2,6,9,G,V,ChordType.MINOR,0
3,9,12,G,VI,ChordType.MAJ_MIN7,0
4,12,15,G,II,ChordType.MINOR,0
...,...,...,...,...,...,...
90,273,276,C,IV,ChordType.MAJOR,0
91,276,279,C,I,ChordType.MAJOR,0
92,279,282,C,V,ChordType.MAJ_MIN7,0
93,282,285,C,I,ChordType.MAJOR,0


In [ ]:
# BACKUP

chord_types = {
"dominant-seventh": ChordType.MAJ_MIN7,
"major": ChordType.MAJOR,
"major-seventh": ChordType.MAJ_MAJ7,
"diminished": ChordType.DIMINISHED,
"augmented": ChordType.AUGMENTED,
"diminshed-seventh": ChordType.DIM7,
"minor": ChordType.MINOR,
"minor-seventh": ChordType.MIN_MIN7,
"Gr+6": ChordType.MAJOR, # ?
"Fr+6": ChordType.MAJOR, # ?
"It+6": ChordType.MAJOR, # ?
"half-diminished-seventh": ChordType.HALF_DIM7,
}

key_changes = [] 


for k in score.flat.getElementsByClass(m21.key.Key):  
    key_changes.append((k.offset, k))  

if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature):
        key_changes.append((ks.offset, ks.asKey()))  
  
key_changes.sort(key=lambda x: x[0])  

# return the actual music21 Key object, not just the string
def get_key_obj_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            return k_obj
    return m21.key.Key('C')  # C as default Key object

# format key str
def format_key_string(k_obj):
    tonic_name = k_obj.tonic.name
    mode = k_obj.mode
    key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
    return key_str.replace('-', 'b').replace('#', '+')  


rows = []

for harmony in score.flat.getElementsByClass("Harmony"):  

    # get chord onset position
    on = Fraction(harmony.offset)
    
    # 1. Get the music21 Key object for this offset
    current_key_obj = get_key_obj_at_offset(on)
    key_str = format_key_string(current_key_obj)
    
    # 2. Calculate the Roman Numeral Degree

    chord_figure = harmony.figure if harmony.figure else str(harmony)
    degree = "unk" # TBD
    inv = 0 # for now ! 
    
    # 3. Get Chord Type mapping
    # harmony.chordKind gives us things like 'major', 'major-seventh', etc.
    harmony_type_str = harmony.chordKind or "major" # default to major ?

    chord_type = chord_types.get(harmony_type_str, "unknown")

    # Append all details to rows (leaving 'off' to be filled by the shift) 
    rows.append([on, None, key_str, degree, chord_type, inv])

# Create DataFrame with explicit column names (IMPORTANT: COLUMN NAMES NEED TO BE DELETED LATER!)
df = pd.DataFrame(rows, columns=["on", "off", "key", "degree", "type", "inv"])

# Shift the 'on' time of the next chord to become the 'off' time of the current chord
df["off"] = df["on"].shift(-1)

# Handle the last chord's 'off' time using its original duration since shift leaves None
if not df.empty:
    last_harmony = score.flat.getElementsByClass("Harmony")[-1]
    df.iloc[-1, df.columns.get_loc("off")] = Fraction(last_harmony.offset + last_harmony.duration.quarterLength)

# df.to_csv("score4990example.csv")

df

In [ ]:
# TESTS


# labels_df = pd.read_csv(
#         label_csv_path,
#         header=None,
#         names=["on", "off", "key", "degree", "type", "inv"],
#         dtype={"degree": str},
#         converters={"on": Fraction, "off": Fraction},
# )
chord_types = {
"D7": ChordType.MAJ_MIN7,
"M": ChordType.MAJOR,
"M7": ChordType.MAJ_MAJ7,
"d": ChordType.DIMINISHED,
"a": ChordType.AUGMENTED,
"d7": ChordType.DIM7,
"m": ChordType.MINOR,
"m7": ChordType.MIN_MIN7,
"Gr+6": ChordType.MAJOR,
"Fr+6": ChordType.MAJOR,
"It+6": ChordType.MAJOR,
"h7": ChordType.HALF_DIM7,
}

# example score
score = m21.converter.parse("choro_corpus/with_chords/score_4990-Menino_de_Ouro-Ernesto_Nazareth.xml")

key_changes = []  

# .flat handles the timeline alignment across all parts safely
for k in score.flat.getElementsByClass(m21.key.Key):  
    key_changes.append((k.offset, k))  

if not key_changes:
    for ks in score.flat.getElementsByClass(m21.key.KeySignature):
        # .asKey() forces '1 flat' to become an F Major Key object
        key_changes.append((ks.offset, ks.asKey()))  
  
# sort by offset  
key_changes.sort(key=lambda x: x[0])  
  
# Function to get key at a given offset  
def get_key_at_offset(offset):  
    for ks_offset, k_obj in reversed(key_changes):  
        if offset >= ks_offset:  
            # k_obj is now a Key object, so it has a .tonic (Note) and .mode ('major'/'minor')
            tonic_name = k_obj.tonic.name # e.g., 'C', 'F', 'G#'
            mode = k_obj.mode
            # Convert to required format: upper case for major, lower for minor  
            key_str = tonic_name.upper() if mode == 'major' else tonic_name.lower()  
            
            # replace flats/sharps with b/+  
            key_str = key_str.replace('-', 'b').replace('#', '+')  
            return key_str  
            
    return "C"  # default

rows = []  
# Using .flat here ensures the Harmony offsets match the Key offsets 
for harmony in score.flat.getElementsByClass("Harmony"):  
    on_before = None
    on = Fraction(harmony.offset)
    off = Fraction(on + harmony.duration.quarterLength)


    key = get_key_at_offset(on)
    rows.append([on, off, key])

df = pd.DataFrame(rows)

# shift to get chord offset position 
df[1] = df[0].shift(-1)

df
# Disambiguate between major and minor (A-major; fsharp-minor, etc.)
# TBD: "degree", "type", "inv"

